In [4]:
import requests

# A Daraz Nepal search page. Adding ?ajax=true *should* make Daraz return
# the listing as JSON (structured data) instead of the HTML webpage.
# You can change "headphones" to any search term later.
SEARCH_URL = "https://www.daraz.com.np/catalog/?ajax=true&q=headphones"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/json",
}

resp = requests.get(SEARCH_URL, headers=headers, timeout=15)
print("HTTP status:", resp.status_code)
print("Content-Type:", resp.headers.get("Content-Type"))

try:
    data = resp.json()
    print("\nGot JSON. Top-level keys:")
    print(list(data.keys()))
except Exception:
    print("\nNot JSON. First 300 characters of what came back instead:")
    print(resp.text[:300])

HTTP status: 200
Content-Type: application/json; charset=utf-8

Got JSON. Top-level keys:
['templates', 'mods', 'mainInfo', 'seoInfo']


In [5]:
# 'mods' holds the page's modules. The product grid usually lives under
# mods["listItems"]. Let's confirm, and peek at the fields on one product.

mods = data["mods"]
print("Keys inside 'mods':")
print(list(mods.keys()))

items = mods.get("listItems")
if items:
    print(f"\nFound {len(items)} products in 'listItems'.")
    print("\nField names on the first product:")
    print(list(items[0].keys()))
else:
    print("\nNo 'listItems' key found — look at the 'mods' keys above and")
    print("tell me which one sounds like the product list.")

Keys inside 'mods':
['filter', 'listItems', 'breadcrumb', 'sortBar', 'resultTips', 'linksInfo']

Found 40 products in 'listItems'.

Field names on the first product:
['name', 'nid', 'itemId', 'icons', 'image', 'isSmartImage', 'utLogMap', 'originalPriceShow', 'priceShow', 'discount', 'ratingScore', 'review', 'location', 'description', 'thumbs', 'sellerName', 'sellerId', 'brandName', 'brandId', 'cheapest_sku', 'skuId', 'sku', 'categories', 'price', 'inStock', 'originalPrice', 'clickTrace', 'itemSoldCntShow', 'longImageDisplayable', 'skus', 'promotionId', 'isSponsored', 'tItemType', 'skuType', 'adFlag', 'directSimilarUrl', 'gridTitleLine', 'isFission', 'isBadgeAutoScroll', 'showCart', 'showBackIcon', 'showUnitPrice', 'itemUrl', 'querystring']


In [6]:
# Pull the useful bits from each product on this page.
# 'review' = how many reviews the product has; we'll use it to skip
# products with nothing to scrape.

products = []
for it in mods["listItems"]:
    products.append({
        "itemId":      it.get("itemId"),
        "name":        it.get("name"),
        "review":      it.get("review"),
        "ratingScore": it.get("ratingScore"),
    })

print(f"Extracted {len(products)} products.\n")
for p in products[:5]:
    print(p)

Extracted 40 products.

{'itemId': '298448087', 'name': 'P9 Wireless Bluetooth Headphones latest With Stereo Headset', 'review': '166', 'ratingScore': '4.174698795180723'}
{'itemId': '128231523', 'name': 'Wooyu G18 RGB Light Gaming Headset Headphone with Built-in Microphone and Volume Control', 'review': '98', 'ratingScore': '4.26530612244898'}
{'itemId': '347333885', 'name': 'Tune 760 NC, Wireless Over Ear Active Noise Cancellation Headphones with Mic, Upto 50 Hours Playtime, Multi-Device Connectivity, Pure Bass, AUX & Voice Assistant Support for Mobile Phones', 'review': '28', 'ratingScore': '4.071428571428571'}
{'itemId': '457828894', 'name': 'P9 Wireless Bluetooth Headphones With Mix Color Latest with Stereo Headset', 'review': '155', 'ratingScore': '3.806451612903226'}
{'itemId': '123763993', 'name': 'P9 Wireless Bluetooth Headset', 'review': '247', 'ratingScore': '3.979757085020243'}


In [7]:
import time

API_SEARCH = "https://www.daraz.com.np/catalog/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/json",
}

def get_search_products(query, page=1):
    """Fetch one page of Daraz search results for a query term.
    Returns a list of products with itemId, name, review count (int),
    ratingScore (float), and the query they came from."""
    params = {"ajax": "true", "q": query, "page": page}
    resp = requests.get(API_SEARCH, params=params, headers=HEADERS, timeout=15)
    if resp.status_code != 200:
        print(f"  page {page}: HTTP {resp.status_code}")
        return []
    items = (resp.json().get("mods") or {}).get("listItems", [])
    products = []
    for it in items:
        products.append({
            "itemId":      it.get("itemId"),
            "name":        it.get("name"),
            "review":      int(it.get("review") or 0),
            "ratingScore": float(it.get("ratingScore") or 0),
            "query":       query,
        })
    return products

# Test: pull page 1 and page 2 of "headphones", check they're different.
p1 = get_search_products("headphones", page=1)
time.sleep(1)                       # polite pause between requests
p2 = get_search_products("headphones", page=2)

ids1 = {p["itemId"] for p in p1}
ids2 = {p["itemId"] for p in p2}
print(f"Page 1: {len(p1)} products")
print(f"Page 2: {len(p2)} products")
print(f"itemIds appearing on BOTH pages: {len(ids1 & ids2)}")

Page 1: 40 products
Page 2: 40 products
itemIds appearing on BOTH pages: 0


In [8]:
import time

# Diverse e-commerce categories for variety. Expand this list later for more volume.
SEARCH_TERMS = [
    "headphones", "tshirt", "shoes", "watch", "bag",
    "lipstick", "rice cooker", "mobile cover", "backpack", "perfume",
]
PAGES_PER_TERM = 3          # 40 products per page → ~120 products per term

all_products = []
for term in SEARCH_TERMS:
    for page in range(1, PAGES_PER_TERM + 1):
        batch = get_search_products(term, page=page)
        all_products.extend(batch)
        print(f"{term!r} page {page}: {len(batch)} (running total {len(all_products)})")
        time.sleep(1)        # polite pause between requests

# De-duplicate by itemId: same product can appear under different searches.
unique = {}
for p in all_products:
    unique[p["itemId"]] = p
products_pool = list(unique.values())

with_reviews = [p for p in products_pool if p["review"] > 0]

print(f"\nCollected {len(all_products)} products total.")
print(f"Unique products after de-duplication: {len(products_pool)}")
print(f"Of those, {len(with_reviews)} have at least one review.")

'headphones' page 1: 40 (running total 40)
'headphones' page 2: 40 (running total 80)
'headphones' page 3: 40 (running total 120)
'tshirt' page 1: 40 (running total 160)
'tshirt' page 2: 40 (running total 200)
'tshirt' page 3: 40 (running total 240)
'shoes' page 1: 40 (running total 280)
'shoes' page 2: 40 (running total 320)
'shoes' page 3: 40 (running total 360)
'watch' page 1: 40 (running total 400)
'watch' page 2: 40 (running total 440)
'watch' page 3: 40 (running total 480)
'bag' page 1: 40 (running total 520)
'bag' page 2: 40 (running total 560)
'bag' page 3: 40 (running total 600)
'lipstick' page 1: 40 (running total 640)
'lipstick' page 2: 40 (running total 680)
'lipstick' page 3: 40 (running total 720)
'rice cooker' page 1: 40 (running total 760)
'rice cooker' page 2: 40 (running total 800)
'rice cooker' page 3: 40 (running total 840)
'mobile cover' page 1: 40 (running total 880)
'mobile cover' page 2: 40 (running total 920)
'mobile cover' page 3: 40 (running total 960)
'backp

In [9]:
REVIEW_API = "https://my.daraz.com.np/pdp/review/getReviewList"

def get_reviews(item_id, max_pages=3):
    """Fetch reviews for one product by itemId.
    Returns dicts: itemId, rating, review_text, review_date, source."""
    reviews = []
    for page in range(1, max_pages + 1):
        params = {"itemId": item_id, "pageSize": 10, "filter": 0, "sort": 0, "pageNo": page}
        resp = requests.get(REVIEW_API, params=params, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            break
        items = (resp.json().get("model") or {}).get("items", [])
        if not items:                      # no more reviews on later pages
            break
        for it in items:
            reviews.append({
                "itemId":      item_id,
                "rating":      it.get("rating"),
                "review_text": (it.get("reviewContent") or "").strip(),
                "review_date": it.get("reviewTime"),
                "source":      "daraz",
            })
        time.sleep(1)                      # polite pause
    return reviews

# Test on the product with the most reviews in our pool.
test_product = max(with_reviews, key=lambda p: p["review"])
print(f"Testing on: {test_product['name'][:60]}")
print(f"itemId {test_product['itemId']}, claims {test_product['review']} reviews\n")

sample = get_reviews(test_product["itemId"], max_pages=3)
print(f"Pulled {len(sample)} reviews.\n")
for r in sample[:5]:
    print(f"[{r['rating']}*] {r['review_text'][:100]}")

Testing on: Slim Laptop Backpack With USB Charging Port Bag  for Men and
itemId 108870858, claims 969 reviews

Pulled 30 reviews.

[4*] The product is good But size is enough
[5*] very very good laptop bag at this price. Comfortable and quality product. picture ma jasto dekhako x
[5*] जस्तो सोचेर अर्डर गरेको त्यो भन्दा अझ धेरै राम्रो पाय। डेलिभरी पनी १ दिन मै आयो। धेरै धेरै धन्य बाद 
[5*] it's really good 👍 price anusar bag akdamai ramro xa. thankyou 😊
[4*] मैले सोचे जस्तै ब्याग  आयो खुशी छु । बिक्रेतालाई धन्यवाद । फेरि पनि मूल्य अनुसारको राम्रो लाग्यो ।  


In [13]:
import csv

# Scrape richest products first — fastest route to volume and to the rare
# neutral/negative classes.
products_to_scrape = sorted(with_reviews, key=lambda p: p["review"], reverse=True)

LIMIT = 25            # start small to confirm; raise to len(products_to_scrape) for the real run
SAVE_EVERY = 25       # checkpoint to disk every N products (protects a long run from crashes)
OUTPUT_CSV = "daraz_reviews_raw.csv"

def save_csv(rows, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["itemId", "rating", "review_text", "review_date", "source"])
        w.writeheader()
        w.writerows(rows)

collected = []
for i, prod in enumerate(products_to_scrape[:LIMIT], start=1):
    try:
        collected.extend(get_reviews(prod["itemId"], max_pages=3))
    except Exception as e:
        print(f"  product {prod['itemId']} failed: {e}")   # skip, don't crash
    if i % 5 == 0 or i == LIMIT:
        print(f"{i}/{LIMIT} products, {len(collected)} reviews so far")
    if i % SAVE_EVERY == 0:
        save_csv(collected, OUTPUT_CSV)

save_csv(collected, OUTPUT_CSV)
print(f"\nDone. {len(collected)} reviews from {LIMIT} products → {OUTPUT_CSV}")

5/25 products, 150 reviews so far
10/25 products, 300 reviews so far
15/25 products, 450 reviews so far
20/25 products, 600 reviews so far
25/25 products, 750 reviews so far

Done. 750 reviews from 25 products → daraz_reviews_raw.csv


In [16]:
import pandas as pd

df = pd.read_csv("daraz_reviews_raw.csv")
print(f"Total reviews pulled: {len(df)}\n")
print("Reviews per star rating:")
print(df["rating"].value_counts().sort_index())

Total reviews pulled: 750

Reviews per star rating:
rating
1      8
2      1
3     11
4    100
5    630
Name: count, dtype: int64


In [17]:
# Check whether the review API exposes a star-rating filter,
# so we can pull negatives directly instead of scraping thousands.
params = {"itemId": "108870858", "pageSize": 10, "filter": 0, "sort": 0, "pageNo": 1}
resp = requests.get(REVIEW_API, params=params, headers=HEADERS, timeout=15)
model = resp.json().get("model", {})

print("Keys inside 'model':")
print(list(model.keys()))

Keys inside 'model':
['items', 'paging', 'item', 'ratings', 'mediaList', 'showInfo', 'pageVersion']


In [18]:
import json
print("ratings:")
print(json.dumps(model["ratings"], indent=2, ensure_ascii=False))
print("\npaging:")
print(json.dumps(model["paging"], indent=2, ensure_ascii=False))

ratings:
{
  "average": 4.4,
  "rateCount": 969,
  "reviewCount": 969,
  "imagesCount": 179,
  "videosCount": 3,
  "withImageCount": 151,
  "withVideoCount": 3,
  "localReviewsCount": 969,
  "withMediaCount": 156,
  "rate12Stars": 65,
  "hiddenCount": 394,
  "scores": [
    658,
    174,
    70,
    23,
    42
  ],
  "mediaNum": 127,
  "labelCounts": [],
  "impressionTags": null
}

paging:
{
  "totalItems": 969,
  "totalPages": 97,
  "currentPage": 1
}


In [19]:
# Discover which 'sort' value brings low-rated reviews to the top.
for s in range(0, 7):
    params = {"itemId": "108870858", "pageSize": 10, "filter": 0, "sort": s, "pageNo": 1}
    resp = requests.get(REVIEW_API, params=params, headers=HEADERS, timeout=15)
    items = (resp.json().get("model") or {}).get("items", [])
    ratings = [it.get("rating") for it in items]
    print(f"sort={s}: first-page ratings = {ratings}")
    time.sleep(1)

sort=0: first-page ratings = [4, 5, 5, 5, 4, 5, 5, 5, 5, 4]
sort=1: first-page ratings = [5, 5, 5, 5, 5, 5, 5, 4, 5, 2]
sort=2: first-page ratings = [5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
sort=3: first-page ratings = [5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
sort=4: first-page ratings = [4, 5, 5, 5, 4, 5, 5, 5, 5, 4]
sort=5: first-page ratings = [4, 5, 5, 5, 5, 5, 4, 4, 5, 5]
sort=6: first-page ratings = [4, 5, 5, 5, 5, 5, 4, 4, 5, 5]


In [20]:
# Try the 'filter' parameter to restrict reviews to specific star ratings.
for fl in range(0, 8):
    params = {"itemId": "108870858", "pageSize": 10, "filter": fl, "sort": 0, "pageNo": 1}
    resp = requests.get(REVIEW_API, params=params, headers=HEADERS, timeout=15)
    items = (resp.json().get("model") or {}).get("items", [])
    ratings = [it.get("rating") for it in items]
    print(f"filter={fl}: {len(items)} reviews, ratings = {ratings}")
    time.sleep(1)

filter=0: 10 reviews, ratings = [4, 5, 5, 5, 4, 5, 5, 5, 5, 4]
filter=1: 10 reviews, ratings = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
filter=2: 10 reviews, ratings = [1, 1, 1, 2, 2, 1, 2, 1, 2, 2]
filter=3: 10 reviews, ratings = [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
filter=4: 10 reviews, ratings = [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
filter=5: 10 reviews, ratings = [5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
filter=6: 0 reviews, ratings = []
filter=7: 0 reviews, ratings = []


In [21]:
from collections import Counter

def get_reviews_filtered(item_id, filter_code, max_pages=30):
    """Pull reviews for one product, restricted to a star-filter, across pages."""
    out = []
    for page in range(1, max_pages + 1):
        params = {"itemId": item_id, "pageSize": 10, "filter": filter_code, "sort": 0, "pageNo": page}
        resp = requests.get(REVIEW_API, params=params, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            break
        items = (resp.json().get("model") or {}).get("items", [])
        if not items:                       # ran out of reviews for this filter
            break
        for it in items:
            out.append({"rating": it.get("rating"),
                        "review_text": (it.get("reviewContent") or "").strip()})
        time.sleep(1)
    return out

neg = get_reviews_filtered("108870858", filter_code=2)   # 1–2 star
neu = get_reviews_filtered("108870858", filter_code=3)   # 3 star

print(f"Negatives (filter=2): {len(neg)} reviews, mix = {Counter(r['rating'] for r in neg)}")
print(f"Neutrals  (filter=3): {len(neu)} reviews, mix = {Counter(r['rating'] for r in neu)}")

Negatives (filter=2): 28 reviews, mix = Counter({1: 15, 2: 13})
Neutrals  (filter=3): 25 reviews, mix = Counter({3: 25})


In [22]:
products_sorted = sorted(with_reviews, key=lambda p: p["review"], reverse=True)

def fetch_filter(item_id, filter_code, max_pages):
    """Pull text-bearing reviews for one product at a given star-filter."""
    out = []
    for page in range(1, max_pages + 1):
        params = {"itemId": item_id, "pageSize": 10, "filter": filter_code, "sort": 0, "pageNo": page}
        resp = requests.get(REVIEW_API, params=params, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            break
        items = (resp.json().get("model") or {}).get("items", [])
        if not items:
            break
        for it in items:
            txt = (it.get("reviewContent") or "").strip()
            if txt:                              # keep only reviews that have text
                out.append({"itemId": item_id, "rating": it.get("rating"),
                            "review_text": txt, "review_date": it.get("reviewTime"),
                            "source": "daraz"})
        time.sleep(1)
    return out

TARGET_NEG, TARGET_NEU = 150, 150        # modest first run; we'll raise these later
neg, neu = [], []

for i, prod in enumerate(products_sorted, start=1):
    if len(neg) < TARGET_NEG:
        neg += fetch_filter(prod["itemId"], 2, max_pages=5)   # negatives (1–2★)
    if len(neu) < TARGET_NEU:
        neu += fetch_filter(prod["itemId"], 3, max_pages=5)   # neutrals (3★)
    if i % 10 == 0:
        print(f"{i} products | negatives {len(neg)} | neutrals {len(neu)}")
    if len(neg) >= TARGET_NEG and len(neu) >= TARGET_NEU:
        print(f"\nHit targets after {i} products."); break

print(f"\nNegatives: {len(neg)} | Neutrals: {len(neu)}")

10 products | negatives 130 | neutrals 140

Hit targets after 12 products.

Negatives: 168 | Neutrals: 159


In [23]:
import os, csv
from collections import Counter

# --- save location: works whether your notebook's cwd is the project root or notebooks/ ---
OUT_DIR = "data/raw/scraped_reviews" if os.path.isdir("data") else "../data/raw/scraped_reviews"
os.makedirs(OUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUT_DIR, "scraped_reviews.csv")
print("Will save to:", os.path.abspath(OUTPUT_CSV), "\n")

# --- targets (raise TARGET_POS toward 3000 later if you want a bigger dataset) ---
TARGET_NEG, TARGET_NEU, TARGET_POS = 600, 600, 1500

def label(r):
    return "negative" if r <= 2 else ("neutral" if r == 3 else "positive")

def save(rows):
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["itemId","rating","review_text","review_date","source","sentiment_label"])
        w.writeheader(); w.writerows(rows)

neg, neu, pos = [], [], []
for i, prod in enumerate(products_sorted, start=1):
    iid = prod["itemId"]
    if len(neg) < TARGET_NEG: neg += fetch_filter(iid, 2, max_pages=5)   # negatives (1–2★)
    if len(neu) < TARGET_NEU: neu += fetch_filter(iid, 3, max_pages=5)   # neutrals (3★)
    if len(pos) < TARGET_POS: pos += fetch_filter(iid, 5, max_pages=3)   # positives (5★)
    if i % 10 == 0:
        print(f"{i} products | neg {len(neg)} | neu {len(neu)} | pos {len(pos)}")
        chk = neg + neu + pos
        for r in chk: r["sentiment_label"] = label(r["rating"])
        save(chk)                                   # checkpoint, crash-safe
    if len(neg) >= TARGET_NEG and len(neu) >= TARGET_NEU and len(pos) >= TARGET_POS:
        print(f"\nAll targets hit after {i} products."); break

# label, then de-duplicate on review text (your synthetic-data lesson: dedupe before splitting)
rows = neg + neu + pos
for r in rows: r["sentiment_label"] = label(r["rating"])
seen, deduped = set(), []
for r in rows:
    key = r["review_text"].strip().lower()
    if key and key not in seen:
        seen.add(key); deduped.append(r)

save(deduped)
print(f"\nCollected {len(rows)} → {len(deduped)} after de-dup")
print("Final class balance:", Counter(r["sentiment_label"] for r in deduped))
print("Saved to:", os.path.abspath(OUTPUT_CSV))

Will save to: C:\Users\user\Desktop\nepali-sentiment-analysis\data\raw\scraped_reviews\scraped_reviews.csv 

10 products | neg 130 | neu 140 | pos 300
20 products | neg 277 | neu 255 | pos 600
30 products | neg 380 | neu 310 | pos 900
40 products | neg 458 | neu 370 | pos 1200
50 products | neg 533 | neu 432 | pos 1496
60 products | neg 604 | neu 479 | pos 1526
70 products | neg 604 | neu 510 | pos 1526
80 products | neg 604 | neu 551 | pos 1526
90 products | neg 604 | neu 576 | pos 1526

All targets hit after 97 products.

Collected 2733 → 2600 after de-dup
Final class balance: Counter({'positive': 1464, 'negative': 587, 'neutral': 549})
Saved to: C:\Users\user\Desktop\nepali-sentiment-analysis\data\raw\scraped_reviews\scraped_reviews.csv


In [24]:
#Spot check
import pandas as pd, os

root = os.path.abspath("." if os.path.isdir("notebooks") else "..")
df = pd.read_csv(os.path.join(root, "data/raw/scraped_reviews/scraped_reviews.csv"))

print("Class totals:", dict(df["sentiment_label"].value_counts()), "\n")

SAMPLE_N = 15
for cls in ["neutral", "negative", "positive"]:      # neutral first — it's the one to scrutinize
    subset = df[df["sentiment_label"] == cls]
    sample = subset.sample(min(SAMPLE_N, len(subset)), random_state=42)
    print(f"\n===== {cls.upper()} =====")
    for n, (_, row) in enumerate(sample.iterrows(), 1):
        text = row["review_text"][:220]
        print(f"{n}. [{row['rating']}★] {text}")

Class totals: {'positive': np.int64(1464), 'negative': np.int64(587), 'neutral': np.int64(549)} 


===== NEUTRAL =====
1. [3★] good for the price
2. [3★] good good
3. [3★] this is not the product I ordered
4. [3★] ramro xa, tara malai chai tauko dukho
5. [3★] Thikai xa price anusar
6. [3★] wrinkle dekhinxa ki jastyo layo tara priz aanusar aaru chai thik xa..sinar ni bhaneko thye tyo chai aayena xa
7. [3★] dekhako jasto hunna huna ta 1199 ma k expect garni but overall good xa pant ma lagauda tik hunxa short ls haru ma lagayo vane cheap quality ko ho vanera tha hunxa
8. [3★] Not a big fan of the smell it's not a sweet smell at least not to me and here for compression ..
9. [3★] It was displayed as a high end product with a significant discount. but it seems  it was freshly made (strong glue smell) locally after the order was placed. so I am not sure if it is exactly the same product, but for th
10. [3★] Credibility is dwindling as shown goods and delivery goods are original and local cop

In [25]:
import requests, time

API_SEARCH = "https://www.daraz.com.np/catalog/"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
           "Accept": "application/json"}

def get_search_products(query, page=1):
    params = {"ajax": "true", "q": query, "page": page}
    resp = requests.get(API_SEARCH, params=params, headers=HEADERS, timeout=15)
    if resp.status_code != 200:
        return []
    items = (resp.json().get("mods") or {}).get("listItems", [])
    return [{"itemId": it.get("itemId"), "name": it.get("name"),
             "review": int(it.get("review") or 0),
             "ratingScore": float(it.get("ratingScore") or 0)} for it in items]

SEARCH_TERMS = [
    # electronics
    "headphones", "earbuds", "power bank", "smart watch", "keyboard", "mouse",
    "bluetooth speaker", "phone charger", "mobile cover", "laptop bag",
    # fashion
    "tshirt", "jeans", "kurta", "shoes", "sandals", "jacket", "watch",
    "sunglasses", "wallet", "backpack", "bag",
    # beauty
    "lipstick", "perfume", "foundation", "sunscreen", "shampoo", "face wash", "nail polish",
    # home & kitchen
    "rice cooker", "blender", "water bottle", "bedsheet", "frying pan", "lunch box",
    # other
    "toys", "notebook", "pen", "umbrella",
]
PAGES_PER_TERM = 3

all_products = []
for term in SEARCH_TERMS:
    for page in range(1, PAGES_PER_TERM + 1):
        all_products.extend(get_search_products(term, page=page))
        time.sleep(1)
    print(f"{term!r} done — running total {len(all_products)}")

unique = {p["itemId"]: p for p in all_products}
products_pool = list(unique.values())
with_reviews = [p for p in products_pool if p["review"] > 0]
products_sorted = sorted(with_reviews, key=lambda p: p["review"], reverse=True)
print(f"\nUnique products: {len(products_pool)} | with reviews: {len(with_reviews)}")

'headphones' done — running total 120
'earbuds' done — running total 240
'power bank' done — running total 360
'smart watch' done — running total 480
'keyboard' done — running total 600
'mouse' done — running total 720
'bluetooth speaker' done — running total 840
'phone charger' done — running total 960
'mobile cover' done — running total 1080
'laptop bag' done — running total 1200
'tshirt' done — running total 1320
'jeans' done — running total 1440
'kurta' done — running total 1560
'shoes' done — running total 1680
'sandals' done — running total 1800
'jacket' done — running total 1920
'watch' done — running total 2040
'sunglasses' done — running total 2160
'wallet' done — running total 2280
'backpack' done — running total 2400
'bag' done — running total 2520
'lipstick' done — running total 2640
'perfume' done — running total 2760
'foundation' done — running total 2880
'sunscreen' done — running total 3000
'shampoo' done — running total 3120
'face wash' done — running total 3240
'nail 

JSONDecodeError: Expecting value: line 2 column 1 (char 2)

In [26]:
# We already gathered ~3,960 products before the hiccup — no need to re-scrape.
unique = {p["itemId"]: p for p in all_products}
products_pool = list(unique.values())
with_reviews = [p for p in products_pool if p["review"] > 0]
products_sorted = sorted(with_reviews, key=lambda p: p["review"], reverse=True)

print(f"Salvaged from {len(all_products)} collected products")
print(f"Unique: {len(products_pool)} | with reviews: {len(with_reviews)}")

Salvaged from 3960 collected products
Unique: 3927 | with reviews: 3700


In [27]:
import os, csv, requests, time
from collections import Counter

REVIEW_API = "https://my.daraz.com.np/pdp/review/getReviewList"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/124.0 Safari/537.36", "Accept": "application/json"}

root = os.path.abspath("." if os.path.isdir("notebooks") else "..")
CSV_PATH = os.path.join(root, "data/raw/scraped_reviews/scraped_reviews.csv")
FIELDS = ["itemId","rating","review_text","review_date","source","sentiment_label"]

# Combined targets — lower these for a quicker run
TARGET_NEG, TARGET_NEU, TARGET_POS = 1200, 1200, 2000

def label(r): return "negative" if r <= 2 else ("neutral" if r == 3 else "positive")

def fetch_filter(item_id, filter_code, max_pages):
    out = []
    for page in range(1, max_pages + 1):
        params = {"itemId": item_id, "pageSize": 10, "filter": filter_code, "sort": 0, "pageNo": page}
        try:
            data = requests.get(REVIEW_API, params=params, headers=HEADERS, timeout=15).json()
        except Exception:
            break                                   # bad/non-JSON response → skip, don't crash
        items = (data.get("model") or {}).get("items", [])
        if not items: break
        for it in items:
            txt = (it.get("reviewContent") or "").strip()
            if txt:
                out.append({"itemId": item_id, "rating": it.get("rating"),
                            "review_text": txt, "review_date": it.get("reviewTime"), "source": "daraz"})
        time.sleep(1)
    return out

# load existing data
existing = []
with open(CSV_PATH, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        row["rating"] = int(row["rating"]); existing.append(row)
done_ids   = {r["itemId"] for r in existing}
seen_text  = {r["review_text"].strip().lower() for r in existing}
counts     = Counter(r["sentiment_label"] for r in existing)
print("Starting from:", dict(counts), f"| {len(done_ids)} products already done\n")

def save(rows):
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS); w.writeheader(); w.writerows(rows)

new_rows = []
need = lambda cls, tgt: counts[cls] < tgt
for i, prod in enumerate(products_sorted, start=1):
    iid = prod["itemId"]
    if iid in done_ids: continue                    # skip already-scraped products
    for code, cls, tgt, pages in [(2,"negative",TARGET_NEG,5),(3,"neutral",TARGET_NEU,5),(5,"positive",TARGET_POS,2)]:
        if need(cls, tgt):
            for r in fetch_filter(iid, code, pages):
                key = r["review_text"].strip().lower()
                if key not in seen_text:
                    seen_text.add(key); r["sentiment_label"] = label(r["rating"])
                    new_rows.append(r); counts[r["sentiment_label"]] += 1
    done_ids.add(iid)
    if i % 20 == 0:
        print(f"{i} scanned | {dict(counts)}"); save(existing + new_rows)   # checkpoint
    if not (need("negative",TARGET_NEG) or need("neutral",TARGET_NEU) or need("positive",TARGET_POS)):
        print(f"\nAll targets hit after scanning {i} products."); break

save(existing + new_rows)
print(f"\nAdded {len(new_rows)} new reviews → {len(existing)+len(new_rows)} total")
print("Final balance:", dict(Counter(r['sentiment_label'] for r in (existing+new_rows))))

Starting from: {'negative': 587, 'neutral': 549, 'positive': 1464} | 95 products already done


All targets hit after scanning 32 products.

Added 1829 new reviews → 4429 total
Final balance: {'negative': 1200, 'neutral': 1210, 'positive': 2019}
